In [1]:
# Step 1: Load dataset
import pandas as pd

df = pd.read_csv('/content/processed_nlp_data.csv')

df.head()

,Customer Age,Customer Gender,Product Purchased,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,...,purchase_day,desc_length,subject_length,cleaned_description,cleaned_subject,final_description,final_subject,lemmatized_description,lemmatized_subject,final_text
0,32,Other,GoPro Hero,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,...,22,284,13,im having an issue with the productpurchased p...,product setup,im issue productpurchased please assist billin...,product setup,im issue productpurchased please assist billin...,product setup,product setup im issue productpurchased please...
1,42,Female,LG Smart TV,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,...,22,282,24,im having an issue with the productpurchased p...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,peripheral compatibility im issue productpurch...
2,48,Other,Dell XPS,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,...,14,275,15,im facing a problem with my productpurchased t...,network problem,im facing problem productpurchased productpurc...,network problem,im facing problem productpurchased productpurc...,network problem,network problem im facing problem productpurch...
3,27,Female,Microsoft Office,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,...,13,262,14,im having an issue with the productpurchased p...,account access,im issue productpurchased please assist proble...,account access,im issue productpurchased please assist proble...,account access,account access im issue productpurchased pleas...
4,67,Female,Autodesk AutoCAD,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,...,4,333,9,im having an issue with the productpurchased p...,data loss,im issue productpurchased please assist note s...,data loss,im issue productpurchased please assist note s...,data loss,data loss im issue productpurchased please ass...


In [2]:
# Step 2: Create better feature using SUBJECT + PRODUCT
df['better_text'] = df['Ticket Subject'] + " " + df['Product Purchased'].fillna('')

df[['better_text', 'Ticket Type']].head()

,better_text,Ticket Type
0,Product setup GoPro Hero,Technical issue
1,Peripheral compatibility LG Smart TV,Technical issue
2,Network problem Dell XPS,Technical issue
3,Account access Microsoft Office,Billing inquiry
4,Data loss Autodesk AutoCAD,Billing inquiry


In [3]:
# Step 3: Features and Target
X = df['better_text']
y = df['Ticket Type']

In [4]:
# Step 4: Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (6775,)
Testing samples: (1694,)


In [5]:
# Step 5: TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [6]:
# Step 6: Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=2000)

In [7]:
# Step 7: Predict
y_pred = model.predict(X_test_tfidf)

In [8]:
# Step 8: Evaluate model
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.18240850059031877

Classification Report:

                      precision    recall  f1-score   support

     Billing inquiry       0.23      0.18      0.20       357
Cancellation request       0.17      0.18      0.18       327
     Product inquiry       0.17      0.15      0.16       316
      Refund request       0.18      0.21      0.19       345
     Technical issue       0.18      0.18      0.18       349

            accuracy                           0.18      1694
           macro avg       0.18      0.18      0.18      1694
        weighted avg       0.18      0.18      0.18      1694



In [9]:
print(df['Ticket Priority'].value_counts())

Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64


In [10]:
# Step 2: Change target variable
X = df['better_text']
y = df['Ticket Priority']

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (6775,)
Testing samples: (1694,)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=2000)

In [14]:
y_pred = model.predict(X_test_tfidf)

In [15]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.22845336481700118

Classification Report:

              precision    recall  f1-score   support

    Critical       0.22      0.24      0.23       411
        High       0.21      0.22      0.21       409
         Low       0.23      0.21      0.22       415
      Medium       0.25      0.24      0.25       459

    accuracy                           0.23      1694
   macro avg       0.23      0.23      0.23      1694
weighted avg       0.23      0.23      0.23      1694



In [16]:
# Step 8: Add structured features

# Select useful columns
df_model = df[['better_text', 'Customer Age', 'Customer Gender', 'Ticket Channel', 'Ticket Priority']]

df_model.head()

,better_text,Customer Age,Customer Gender,Ticket Channel,Ticket Priority
0,Product setup GoPro Hero,32,Other,Social media,Critical
1,Peripheral compatibility LG Smart TV,42,Female,Chat,Critical
2,Network problem Dell XPS,48,Other,Social media,Low
3,Account access Microsoft Office,27,Female,Social media,Low
4,Data loss Autodesk AutoCAD,67,Female,Email,Low


In [18]:
# Convert categorical features
df_model = pd.get_dummies(df_model, columns=['Customer Gender', 'Ticket Channel'], drop_first=True)

In [19]:
X_text = df_model['better_text']
X_extra = df_model.drop(columns=['better_text', 'Ticket Priority'])

y = df_model['Ticket Priority']

In [20]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, X_train_extra, X_test_extra, y_train, y_test = train_test_split(
    X_text, X_extra, y, test_size=0.2, random_state=42
)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

In [25]:
from scipy.sparse import hstack
import numpy as np

# Convert extra features to numeric
X_train_extra = X_train_extra.astype(float)
X_test_extra = X_test_extra.astype(float)

# Combine
X_train_final = hstack([X_train_tfidf, X_train_extra])
X_test_final = hstack([X_test_tfidf, X_test_extra])

In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

model.fit(X_train_final, y_train)

LogisticRegression(max_iter=2000)

In [27]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.2384887839433294

Classification Report:

              precision    recall  f1-score   support

    Critical       0.23      0.26      0.25       411
        High       0.21      0.22      0.22       409
         Low       0.22      0.20      0.21       415
      Medium       0.28      0.27      0.27       459

    accuracy                           0.24      1694
   macro avg       0.24      0.24      0.24      1694
weighted avg       0.24      0.24      0.24      1694



In [28]:
from sklearn.naive_bayes import MultinomialNB

# Create model
model = MultinomialNB()

# Train
model.fit(X_train_final, y_train)

# Predict
y_pred = model.predict(X_test_final)

# Evaluate
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.2449822904368359

Classification Report:

              precision    recall  f1-score   support

    Critical       0.23      0.27      0.25       411
        High       0.22      0.24      0.23       409
         Low       0.23      0.18      0.20       415
      Medium       0.29      0.28      0.28       459

    accuracy                           0.24      1694
   macro avg       0.24      0.24      0.24      1694
weighted avg       0.25      0.24      0.24      1694



In [29]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)

model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.23081463990554899

Classification Report:

              precision    recall  f1-score   support

    Critical       0.24      0.27      0.25       411
        High       0.19      0.19      0.19       409
         Low       0.22      0.20      0.21       415
      Medium       0.27      0.26      0.27       459

    accuracy                           0.23      1694
   macro avg       0.23      0.23      0.23      1694
weighted avg       0.23      0.23      0.23      1694

